# PC-FNO3D réel (Grand Tunis) — le modèle livré en ligne (atom.zelqane.com)

Sert la **v2 prolongée** (`runs/grand_tunis_v2_long/best.pt`, le modèle livré) à l'adresse fixe :

```
navigateur / vue 3D --> atom.zelqane.com (Worker de Rayen, clé injectée côté serveur)
                    --> atom-origin.zelqane.com (tunnel nommé, URL fixe)
                    --> CE carnet : cloudflared --> service sur 127.0.0.1:8000 (GPU)
```

Entrée de production : ERA5 10 m sur le réseau régulier + les **vraies mesures** des 3 stations
(Tunis-Carthage, Zaghouan, Nabeul) + le forçage ERA5 de l'instant. Instants servis : les 180
instants de test 2022 du cache.

**Avant la première exécution** : secret Colab `TUNNEL_CRED_CONTENTS` (identifiants du tunnel
`atom`, remis par Rayen de la main à la main), avec « Accès au notebook ». La clé d'API n'est
présente que par son **empreinte**.

**Exécution** : GPU, puis **Tout exécuter** (~20 min : code, données, instants de test,
volumes 3D, démarrage). Laisse ensuite l'onglet ouvert : le service vit tant que la session vit.

## 1. Drive, code et données

In [ ]:
from google.colab import drive
import os as _os
if not _os.path.isdir("/content/drive/MyDrive"):   # deja monte (panneau Fichiers) : ne pas redemander
    drive.mount("/content/drive")

import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

GT = Path("/content/drive/MyDrive/pcfno3d_grand_tunis")
GT.mkdir(parents=True, exist_ok=True)
JOURNAUX = GT / "journaux"
JOURNAUX.mkdir(exist_ok=True)

# code : archive deposee sur le Drive depuis ines-pc
CODE = Path("/content/PFE")
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
with zipfile.ZipFile(GT / "pcfno3d_code.zip") as zf:
    zf.extractall(CODE)
tampon = CODE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q pandera xarray netCDF4 h5netcdf 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

def lancer(nom: str, commande: str) -> None:
    """Commande obligatoire ; sortie affichee ET journalisee sur le Drive ; arret si echec."""
    t0 = time.time()
    with open(JOURNAUX / f"{nom}.log", "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} : {commande}\n")
        p = subprocess.Popen(commande, shell=True, cwd="/content/PFE", stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for ligne in p.stdout:
            print(ligne, end="")
            log.write(ligne)
            log.flush()
        p.wait()
    print(f"[{nom}] {time.time() - t0:.0f} s, code {p.returncode}")
    if p.returncode:
        raise RuntimeError(f"{nom} a echoue (code {p.returncode}) : voir {JOURNAUX / (nom + '.log')}")

# donnees : grille reelle, forcage ERA5, stations (une archive sur le Drive)
DON = Path("/content/donnees")
if not (DON / "grand_tunis" / "forcage_era5.csv").exists():
    with zipfile.ZipFile(GT / "grand_tunis_donnees.zip") as zf:
        zf.extractall(DON)
# mises a jour deposees apres coup sur le Drive (morceaux de moins de 10 Mo) :
# maj_<fichier> ou maj_<fichier>.000, .001... -> remplacent <fichier> dans donnees/grand_tunis
import collections
morceaux = collections.defaultdict(list)
for f in sorted(GT.glob("maj_*")):
    nom = f.name[4:]
    tete, _, queue = nom.rpartition(".")
    morceaux[tete if queue.isdigit() and tete else nom].append(f)
for nom, fs in morceaux.items():
    with open(DON / "grand_tunis" / nom, "wb") as out:
        for f in sorted(fs):
            out.write(f.read_bytes())
    print("mise a jour :", nom, "<-", [f.name for f in sorted(fs)])
print(sorted(p.name for p in (DON / "grand_tunis").iterdir()))

# le service web : aucune de ces bibliotheques ne touche a numpy
!pip install -q fastapi uvicorn python-multipart httpx mcp==2.3.0 2>&1 | tail -2

## 2. Instants de test (cache) et vraies mesures des stations

In [ ]:
# instants de TEST des 3 parties (S, Y, E) + index fusionne ; observations reelles du domaine
LOC = Path("/content/grand_tunis")
CACHE, JEU = LOC / "cache", LOC / "jeu"
CACHE.mkdir(parents=True, exist_ok=True)
JEU.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(GT / "jeu.zip") as zf:
    zf.extractall(JEU)
import pandas as pd
index, n = [], 0
for k in (1, 2, 3):
    with zipfile.ZipFile(GT / f"cache_partie_{k}.zip") as zf:
        with zf.open(f"index_partie_{k}.csv") as f:
            idx = pd.read_csv(f)
        index.append(idx)
        test = set(idx.loc[idx["split"] == "test", "stem"])
        noms = [x for x in zf.namelist() if x.rsplit("_", 1)[0] in test]
        zf.extractall(CACHE, members=noms)
        n += len(noms)
pd.concat(index).drop_duplicates("stem").to_csv(CACHE / "index.csv", index=False)
OBS = Path("/content/obs")
with zipfile.ZipFile(GT / "grand_tunis_obs.zip") as zf:
    zf.extractall(OBS)
print(f"{n} fichiers de test ; observations : {sorted(p.name for p in OBS.iterdir())}")

## 3. Volumes 3D pour la visualisation

In [ ]:
# volumes 3D de la vue de Rayen (/api/export/<instant>/...) : un pas sur deux a l'horizontale
lancer("export_web", "python -u -m pcfno3d -c grand_tunis_v2_service_colab export-web --split test --stride 2")

## 4. Démarrer le service derrière le tunnel nommé

In [ ]:
from google.colab import userdata
from service.colab import serve_public

TUNNEL_ID = "e552f718-2478-4fe3-82b4-8d33a4b9eb6e"
HOTE = "atom-origin.zelqane.com"
CLE_NOM = "atom-worker"                      # la cle de lecture du Worker de Rayen
CLE_EMPREINTE = "ce7965c54284eb6d1a274119839a71d9785731609d04b17bd08fb6da7d2c30fd"

try:
    identifiants = userdata.get("TUNNEL_CRED_CONTENTS")
except Exception as exc:                     # secret absent, ou acces non accorde
    raise RuntimeError(
        "secret Colab TUNNEL_CRED_CONTENTS introuvable ou non autorise : panneau de gauche, "
        "icone cle, ajoute-le (contenu remis par Rayen) et active « Acces au notebook ».") from None

info = serve_public(config="grand_tunis_v2_service_colab", key_name=CLE_NOM, scope="read", key_hash=CLE_EMPREINTE,
                    quota_minute=1200,
                    named_tunnel={"id": TUNNEL_ID, "hostname": HOTE, "credentials": identifiants})
del identifiants

## 5. Vérifier de l'extérieur

In [ ]:
import urllib.request
req = urllib.request.Request(f"https://{HOTE}/api/health", headers={"User-Agent": "pcfno3d-colab/1.0"})
with urllib.request.urlopen(req, timeout=60) as r:
    sante = json.loads(r.read().decode())
print(json.dumps(sante, ensure_ascii=False, indent=1))
assert sante["status"] == "ok", "le service ne repond pas correctement par le tunnel"
print(f"\nEN LIGNE : https://{HOTE} (v2 prolongee Grand Tunis) — previens Rayen (sans coller de cle).")

## 6. Garder en vie

In [ ]:
try:
    while True:
        tunnel_ok = info["tunnel"].alive
        service_ok = info["process"].poll() is None
        print(time.strftime("%H:%M"), "| tunnel", "ok" if tunnel_ok else "ARRETE",
              "| service", "ok" if service_ok else "ARRETE", flush=True)
        if not (tunnel_ok and service_ok):
            print("journaux :", info["log_path"], "et", info["tunnel"].log_path)
            break
        time.sleep(600)
except KeyboardInterrupt:
    print("surveillance arretee — le service tourne toujours")

In [ ]:
# Garde-fou : « Tout executer » atteint aussi cette cellule des que la surveillance s'arrete.
# Mets ARRETER = True pour couper volontairement.
ARRETER = False
if ARRETER:
    from service.colab import stop
    stop(info)
    print("service et tunnel arretes : atom.zelqane.com repasse en « endormi »")
else:
    print("service laisse en marche (ARRETER = False)")